In [1]:
#imports
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import keras

In [2]:
# Change pixel values to be range [0,1]
def normalize_img(image, label):
	"""Normalizes images: `uint8` -> `float32`."""
	return tf.cast(image, tf.float32) / 255., label

In [3]:
# load data
(ds_in, ds_out) = tfds.load('pneumonia_mnist', split=['train[:50%]', 'train[50%:100%]'], shuffle_files=False, as_supervised=True)

In [4]:
ds_in = ds_in.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)

features = [[] for i in range(10)]
labels = [[] for i in range(10)]

#create subsets for in-models
for step, (x, y) in enumerate(ds_in):
	start_index = step % 10
	for offset in range(5):
		subset_index = (start_index + offset) % 10
		features[subset_index].append(x)
		labels[subset_index].append(y)

subsets = [tf.data.Dataset.from_tensor_slices((features[i], labels[i])) for i in range(len(features))]



In [5]:
#cache and prefetch data for faster performance
ds_in = ds_in.cache()
ds_in = ds_in.batch(64)
ds_in = ds_in.prefetch(tf.data.AUTOTUNE)

for i in range(len(subsets)):
	subsets[i] = subsets[i].cache()
	subsets[i] = subsets[i].batch(64)
	subsets[i] = subsets[i].prefetch(tf.data.AUTOTUNE)



ds_out = ds_out.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_out = ds_out.batch(64)
ds_out = ds_out.cache()
ds_out = ds_out.prefetch(tf.data.AUTOTUNE)

In [6]:
#target model and shadow model architecture
def get_classifier():
	model = tf.keras.models.Sequential([
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu", input_shape=(28, 28, 1)),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Flatten(),
	tf.keras.layers.Dense(64, activation='relu'),
	tf.keras.layers.Dense(2)
	])

	return model

In [7]:
#train target model on in data
target_model = get_classifier()

target_model.compile(
optimizer=tf.keras.optimizers.Adam(0.001),
loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

target_model.fit(
    ds_in,
    epochs=25,
)

C:\Users\Skylar Marosi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.5717 - sparse_categorical_accuracy: 0.7396
Epoch 2/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.5062 - sparse_categorical_accuracy: 0.7434
Epoch 3/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.3491 - sparse_categorical_accuracy: 0.8454
Epoch 4/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2703 - sparse_categorical_accuracy: 0.8879
Epoch 5/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2448 - sparse_categorical_accuracy: 0.8993
Epoch 6/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2260 - sparse_categorical_accuracy: 0.9070
Epoch 7/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2096 - sparse_categorical_accuracy: 0.9125
Epoch 8/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2010 - sparse_categorical_accuracy: 0.9159
Epoch 9/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1934 - sparse_categorical_accuracy: 0.9210
Epoch 10/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.18

In [14]:
#train and save out-models using mixed loss criteria
mse = keras.losses.MeanSquaredError()
ce = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

for i in range(10):
	optimizer = tf.keras.optimizers.Adam(0.001)
	out_model = get_classifier()
	#warmup model with out data
	"""
	for epoch in range(25):
		ds_in = ds_in.shuffle(buffer_size=10)
		for step, (x_batch_in, y_batch_in) in enumerate(ds_in):
			with tf.GradientTape() as tape:
				student_logits = out_model(x_batch_in, training=False)

				loss_value = ce(y_batch_in, student_logits)
				
			grads = tape.gradient(loss_value, out_model.trainable_weights)

			optimizer.apply_gradients(zip(grads, out_model.trainable_weights))
	"""

	best_loss = 1
	#imitative loss training
	for epoch in range(50):
		ds_out = ds_out.shuffle(buffer_size=10)
		for step, (x_batch_out, y_batch_out) in enumerate(ds_out):
			with tf.GradientTape() as tape:
				student_logits = out_model(x_batch_out, training=False)

				#regular_loss = ce(y_batch_out, student_logits)


				teacher_logits = target_model(x_batch_out)

				imitate_loss = mse(teacher_logits, student_logits)

				if (imitate_loss < best_loss):
					best_loss = imitate_loss
					out_model.save_weights(f'./out_models/out_model_{i+1}.weights.h5')

				#half of loss is regular cross entropy, the other half is how far its outputs were from the target model
				loss_value = imitate_loss
				
			grads = tape.gradient(loss_value, out_model.trainable_weights)

			optimizer.apply_gradients(zip(grads, out_model.trainable_weights))
			
	print(f"finished training model {i+1}")



C:\Users\Skylar Marosi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


finished training model 1
finished training model 2
finished training model 3
finished training model 4
finished training model 5
finished training model 6
finished training model 7
finished training model 8
finished training model 9
finished training model 10


In [15]:
#Alternate way of training in models
mse = keras.losses.MeanSquaredError()
ce = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

for i in range(10):
	optimizer = tf.keras.optimizers.Adam(0.001)
	out_model = get_classifier()
	#warmup model with out data
	"""
	for epoch in range(25):
		ds_in = ds_in.shuffle(buffer_size=10)
		for step, (x_batch_in, y_batch_in) in enumerate(ds_in):
			with tf.GradientTape() as tape:
				student_logits = out_model(x_batch_in, training=False)

				loss_value = ce(y_batch_in, student_logits)
				
			grads = tape.gradient(loss_value, out_model.trainable_weights)

			optimizer.apply_gradients(zip(grads, out_model.trainable_weights))
	"""

	best_loss = 1
	#imitative loss training
	for epoch in range(50):
		ds_in = ds_in.shuffle(buffer_size=10)
		for step, (x_batch_out, y_batch_out) in enumerate(ds_in):
			with tf.GradientTape() as tape:
				student_logits = out_model(x_batch_out, training=False)

				#regular_loss = ce(y_batch_out, student_logits)


				teacher_logits = target_model(x_batch_out)

				imitate_loss = mse(teacher_logits, student_logits)

				if (imitate_loss < best_loss):
					best_loss = imitate_loss
					out_model.save_weights(f'./in_models/in_model_{i+1}.weights.h5')

				#half of loss is regular cross entropy, the other half is how far its outputs were from the target model
				loss_value = imitate_loss
				
			grads = tape.gradient(loss_value, out_model.trainable_weights)

			optimizer.apply_gradients(zip(grads, out_model.trainable_weights))
			
	print(f"finished training model {i+1}")

C:\Users\Skylar Marosi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


finished training model 1
finished training model 2
finished training model 3
finished training model 4
finished training model 5
finished training model 6
finished training model 7
finished training model 8
finished training model 9
finished training model 10


In [58]:
#train and save in-models on in data
for i in range(10):
	in_model = get_classifier()

	in_model.compile(
	optimizer=tf.keras.optimizers.Adam(0.001),
	loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
	)

	checkpoint_path = f'./in_models/in_model_{i+1}.keras'

	checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
		filepath=checkpoint_path,
		monitor='loss',     
		mode='min',            
		save_best_only=True,   
		verbose=0              
	)

	in_model.fit(
    subsets[i],
    epochs=50,
	callbacks=[checkpoint_callback]
	)

	#in_model.save_weights(f'./in_models/in_model_{i+1}.weights.h5')

Epoch 1/50


C:\Users\Skylar Marosi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.6109
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.5708
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.5454
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.4648
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.4039
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.3787
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.3296
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.2976
Epoch 9/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.2812
Epoch 10/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.2639
Epoch 11/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.2492
Epoch 12/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.2395
Epoch 13/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.2256
Epoch 14/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.2153
Epoch 15/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.2066
Epoch 16/50
19/

In [16]:
#load imitative models for inference
in_models = []
out_models = []
for i in range(10):
	in_model = get_classifier()
	in_model.load_weights(f'./in_models/in_model_{i+1}.weights.h5')
	#in_model = keras.models.load_model(f'./in_models/in_model_{i+1}.keras')
	in_models.append(in_model)

	out_model = get_classifier()
	out_model.load_weights(f'./out_models/out_model_{i+1}.weights.h5')
	out_models.append(out_model)

In [17]:
#calculate scaled confidence score for a batch
def scaled_confidence_score(x, y, model):
	pred = model.predict(x)

	confidences = []
	#calculate scaled confidence score for model(x[i])
	for i in range(len(x)):
		probas = tf.nn.softmax(pred[i])
		real = probas[y[i]] + 1e-45
		other = probas[1 - y[i]] + 1e-45
		phi = np.log10(real) - np.log10(other)
		confidences.append(phi)
		#confidences.append(confidence)

	return np.array(confidences)

In [18]:
#make predictions about data
def do_inference(target_model, in_models, out_models, ds, membership):
	correct = 0
	for step, (x_batch, y_batch) in enumerate(ds):
		target_phi = scaled_confidence_score(x_batch, y_batch, target_model)

		mean_out_phi = np.zeros(len(x_batch))
		for i in range(len(out_models)):
			#scaled_confidence_score returns an array where each entry is the model's confidence for that input in the batch, i.e. length 64
			mean_out_phi = mean_out_phi + scaled_confidence_score(x_batch, y_batch, out_models[i])

		#array of length 64, each entry of which is the mean confidence score between the 10 out models for an entry of x_batch_in
		mean_out_phi = mean_out_phi / len(x_batch)

		mean_in_phi = np.zeros(len(x_batch))
		for i in range(len(in_models)):
			#scaled_confidence_score returns an array where each entry is the model's confidence for that input in the batch, i.e. length 64
			mean_in_phi = mean_in_phi + scaled_confidence_score(x_batch, y_batch, in_models[i])

		#array of length 64, each entry of which is the mean confidence score between the 10 in models for an entry of x_batch_in
		mean_in_phi = mean_in_phi / len(x_batch)

		out_dist = (target_phi - mean_out_phi)**2
		in_dist = (target_phi - mean_in_phi)**2

		# if lambda > 0 -> predict "in"
		for i in range(len(x_batch)):
			if (membership):
				if (in_dist[i] < out_dist[i]):
					correct += 1
			else:
				if (out_dist[i] < in_dist[i]):
					correct += 1

	total_entries = sum(batch[0].shape[0] for batch in ds)
	advantage = correct / total_entries
	return advantage


# if lambda > 0 -> predict "in"

In [19]:
in_advantage = do_inference(target_model, in_models, out_models, ds_in, 1)
out_advantage = do_inference(target_model, in_models, out_models, ds_out, 0)

advantage = (in_advantage + out_advantage)/2

print(f'Advantage on in data is {in_advantage}. \nAdvantage on out data is {out_advantage}. \nOverall advantage is {advantage}.')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━